# Build Your Agent
### IEEE SMC · MIT Bengaluru · Technical Symposium · 5 Sep 2026

You just spent an hour watching agent patterns run inside a browser demo — a scripted trace, pre-written, always ending the same way.

This notebook is the same eight patterns, except the model is **real** (Google Gemini, free tier) and the failures are **real** — if your tool is described badly, it *will* misfire, live, in front of you. That's the point.

Every section below maps directly onto a stage you already saw in **Campus Agent Lab**, or a field you already filled in **Life Agent Builder**. Nothing here is new material — it's the same architecture, now wired to something that actually thinks.

**What you need:** a free Google account and five minutes. No GPU, no billing, no install beyond one `pip` line.

| Notebook section | Mirrors |
|---|---|
| 1 — Plain call | Campus Agent Lab · Stage 01 (Basic LLM) |
| 2 — Give it tools | Campus Agent Lab · Stage 03 (Tool-Calling) |
| 3 — Catch it lying | Campus Agent Lab · Stage 07 (Evaluator) |
| 4 — Add a rail | Campus Agent Lab · Stage 08 (Human-in-Loop) |
| 5 — Capstone | Life Agent Builder · Sense / Decide / Act / Rail |

## Step 0 — Get a free API key (2 minutes, do this first)

1. Go to **[aistudio.google.com](https://aistudio.google.com/apikey)** and sign in with any Google account.
2. Click **Create API key**. Copy it — it's a long string starting with `AIza`.
3. This is free. No credit card. The free tier has a daily request cap, which is more than enough for this notebook.
4. **Never commit this key to a public GitHub repo or paste it into chat.** If you're sharing this notebook publicly, use one of the two safe options in the next cell instead of pasting your key directly into the file.

**In Google Colab (recommended):** click the key icon (🔑) in the left sidebar → *Secrets* → add a secret named `GEMINI_API_KEY` → paste your key as the value → toggle *Notebook access* on. The code below will find it automatically.

**Running locally / no Colab secrets:** the fallback cell below will just ask you to paste the key when you run it — it is not saved anywhere.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os

api_key = None

# Try Colab secrets first (recommended, keeps the key out of the notebook file)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

# Fallback: ask for it interactively (works outside Colab too, not saved to disk)
if not api_key:
    import getpass
    api_key = getpass.getpass("Paste your Gemini API key (input hidden): ")

os.environ["GEMINI_API_KEY"] = api_key
print("Key loaded." if api_key else "No key found — go back to Step 0.")

In [ ]:
from google import genai

client = genai.Client()

# Model names on the free tier move fast. This is current as of the symposium —
# if it 404s for you, run the cell below to see what's actually live right now
# and swap the string here.
MODEL = "gemini-3.8-flash"

try:
    _test = client.interactions.create(model=MODEL, input="Reply with just the word OK.")
    print(f"✓ Connected. Model '{MODEL}' responded: {_test.output_text.strip()}")
except Exception as e:
    print(f"✗ '{MODEL}' didn't work ({e}).\nRun the next cell to list models that ARE available to your key, then update MODEL above and re-run this cell.")

In [ ]:
# Only run this if the connection test above failed.
# It lists every model your API key can actually use, right now.
for m in client.models.list():
    print(m.name)

---
## 1 — A plain call (mirrors Stage 01 · Basic LLM)

You saw this in the browser: a model with no tools, no access to your timetable, no access to anything real — just text in, text out.

Ask it something it genuinely cannot know, and watch what it does instead of saying "I don't know."

In [ ]:
response = client.interactions.create(
    model=MODEL,
    input="What time is my Robotics Lab workshop this Saturday, and is there a scheduling conflict with anything else on my calendar?",
)
print(response.output_text)

Read what you got back. It almost certainly sounds confident and helpful — and it is **completely made up**, because the model has no calendar, no workshop list, nothing. This is the failure mode from the deck: *"your agent will not throw an exception, it will lie politely."* You just watched it happen live, not in a pre-scripted trace.

Fix: give it real data to work from.

---
## 2 — Give it tools (mirrors Stage 03 · Tool-Calling)

Same two toy functions the browser demo used: a timetable and a conflict checker. This time they're real Python, and the model actually calls them.

In [ ]:
# ---- real (toy) data, same shape as the Campus Agent Lab demo ----
TIMETABLE = {
    "tue": [{"time": "19:00", "event": "Algorithm Lab"}],
    "sat": [],  # Saturday is free
}
WORKSHOPS = {
    "robotics lab": {"day": "sat", "time": "16:00"},
}

def get_timetable(day: str):
    """Returns the student's existing calendar events for a given day (mon/tue/.../sun)."""
    return {"day": day, "events": TIMETABLE.get(day.lower(), [])}

def check_conflict(day: str, time: str):
    """Checks whether a proposed day+time collides with an existing calendar event."""
    events = TIMETABLE.get(day.lower(), [])
    clash = next((e for e in events if e["time"] == time), None)
    return {"conflict": clash is not None, "clashing_event": clash}

available_functions = {
    "get_timetable": get_timetable,
    "check_conflict": check_conflict,
}

tools = [
    {
        "type": "function",
        "name": "get_timetable",
        "description": "Returns the student's existing calendar events for a given day.",
        "parameters": {
            "type": "object",
            "properties": {"day": {"type": "string", "description": "e.g. 'sat', 'tue'"}},
            "required": ["day"],
        },
    },
    {
        "type": "function",
        "name": "check_conflict",
        "description": "Checks whether a proposed day+time collides with an existing calendar event.",
        "parameters": {
            "type": "object",
            "properties": {
                "day": {"type": "string"},
                "time": {"type": "string", "description": "24h format, e.g. '19:00'"},
            },
            "required": ["day", "time"],
        },
    },
]

print("Tools defined: get_timetable, check_conflict")

In [ ]:
stage3 = client.interactions.create(
    model=MODEL,
    input=(
        "I want to register for the Robotics Lab workshop on Saturday at 4pm. "
        "Check my calendar first and tell me if there's a conflict."
    ),
    tools=tools,
)

for step in stage3.steps:
    if step.type == "function_call":
        result = available_functions[step.name](**step.arguments)
        print(f"→ model called {step.name}({step.arguments}) → {result}")

print("\nFinal answer:\n", stage3.output_text)

Notice the model didn't already know your calendar — it **asked its tools** before answering. That's the whole jump from Stage 01 to Stage 03: an agent isn't a smarter prompt, it's a loop that calls out for real data instead of guessing.

Now try something meaner: ask about **Tuesday at 7pm** instead of Saturday — the exact clash from the deck's evaluator slide — and see whether it actually catches its own conflict, or just answers anyway.

In [ ]:
stage3b = client.interactions.create(
    model=MODEL,
    input="I want to register for a workshop on Tuesday at 7pm. Check my calendar and tell me if that works.",
    tools=tools,
)
for step in stage3b.steps:
    if step.type == "function_call":
        result = available_functions[step.name](**step.arguments)
        print(f"→ model called {step.name}({step.arguments}) → {result}")
print("\nFinal answer:\n", stage3b.output_text)

Did it say yes anyway, or did it catch the clash? Either is a realistic outcome — that's exactly the point of the next section. **Having the right tool call is not the same as reasoning correctly about the result.** Don't trust the model to police itself. Add a check that does it for you, in code, every time.

---
## 3 — Catch it lying (mirrors Stage 07 · Evaluator)

This is the slide from the deck: *"the tool call succeeded, it booked the clash anyway."* An evaluator is not another model call — it's ordinary code that checks the answer against ground truth, and forces a retry if it fails.

In [ ]:
def evaluate(day: str, time: str) -> dict:
    """Ground-truth check, independent of anything the model said."""
    check = check_conflict(day, time)
    if check["conflict"]:
        return {"pass": False, "reason": f"{day} {time} clashes with {check['clashing_event']['event']}"}
    return {"pass": True, "reason": "no conflict"}


def run_with_evaluator(day: str, time: str, max_retries: int = 1):
    prev_id = None
    request_text = f"I want to register for a workshop on {day} at {time}. Check my calendar and tell me if that works."

    for attempt in range(max_retries + 1):
        kwargs = {"model": MODEL, "input": request_text, "tools": tools}
        if prev_id:
            kwargs["previous_interaction_id"] = prev_id
        try:
            result = client.interactions.create(**kwargs)
        except TypeError:
            # some SDK versions don't support previous_interaction_id — fall back to a fresh call
            result = client.interactions.create(model=MODEL, input=request_text, tools=tools)

        for step in result.steps:
            if step.type == "function_call":
                available_functions[step.name](**step.arguments)

        check = evaluate(day, time)
        print(f"Attempt {attempt+1}: model said → {result.output_text.strip()[:120]}...")
        print(f"  Evaluator: {'PASS' if check['pass'] else 'FAIL — ' + check['reason']}")

        if check["pass"] or attempt == max_retries:
            return result, check

        prev_id = getattr(result, "id", None)
        request_text = (
            f"That's wrong — {check['reason']}. Do not confirm that booking. "
            f"Suggest a different time that has no conflict."
        )


final_result, final_check = run_with_evaluator("tue", "19:00")

This is the difference between *having* an evaluator and *wiring* one — the phrase from Field Report 03 in the deck, about the alarm that fired and nobody escalated it. Here, the evaluator's FAIL isn't a log line. It goes straight back into the next request and forces a different answer.

---
## 4 — Add a rail (mirrors Stage 08 · Human-in-the-Loop)

Read-only calls (`get_timetable`, `check_conflict`) ran automatically above — no harm in being wrong, you just re-run them. Registering a student is a **write**. Gate it on reversibility, not on how confident the model sounds.

In [ ]:
def register_workshop(day: str, time: str, name: str):
    """WRITE action — this is stage 08's gated tool, not auto-executed."""
    print(f"\n>>> About to register '{name}' for {day} {time}.")
    approve = input(">>> Approve this booking? (y/n): ").strip().lower()
    if approve == "y":
        print(f">>> ✓ Registered {name} for {day} {time}.")
        return True
    print(">>> ✗ Cancelled — no booking made.")
    return False

if final_check["pass"]:
    register_workshop("tue", "19:00", "you")
else:
    print("Evaluator never passed — correctly refusing to even offer the registration step.")

That `input()` call is the entire pattern. Nothing fancier is needed — the point of Stage 08 was never that approval gates are technically hard, it's that people forget to put them in front of the one function that can't be undone.

---
## 5 — Capstone: build your own (mirrors Life Agent Builder)

Pick one pattern from your own life — the same list from the browser tool:

- **The 2am scroll** — screen time past a threshold at night
- **The 11pm cart** — a big purchase, late, unreviewed
- **The 9-tab explosion** — too many open browser tabs, no cleanup
- **"I'll reply properly later"** — a message read but never answered
- **The forgotten deadline** — a recurring task, no completion signal
- or bring your own

Fill in the four functions below — the same SENSE / DECIDE / ACT / RAIL fields from the canvas — except this time `decide()` makes a real call to Gemini.

In [ ]:
def sense():
    """TODO — return a dict describing the situation you're detecting.
    Example for the 2am scroll: {"screen_time_min": 95, "hour": 1, "next_alarm": "07:00"}
    """
    return {"screen_time_min": 95, "hour": 1, "next_alarm": "07:00"}


def decide(state: dict) -> str:
    """TODO — describe your situation and ask Gemini what the agent should say/do.
    This is the only step that calls the model — sense() and act() are plain code,
    exactly like the real tool.
    """
    prompt = (
        f"Current state: {state}. "
        f"Write a short, kind, one-sentence nudge for someone still scrolling at 1am "
        f"with a 7am alarm. Do not lecture. Do not use guilt."
    )
    result = client.interactions.create(model=MODEL, input=prompt)
    return result.output_text.strip()


def act(message: str):
    """TODO — what does the agent actually DO with the decision?
    Keep it to a notification-style action, not a destructive one.
    """
    print(f"📱 Notification: {message}")


def rail(state: dict) -> bool:
    """TODO — the one thing this agent must NEVER do.
    Return False to block act() from running at all.
    Example: never fire more than once a night, never fire before 9pm.
    """
    return state["hour"] >= 21 or state["hour"] <= 5


# ---- the loop — same shape every agent in this notebook has used ----
state = sense()
if rail(state):
    message = decide(state)
    act(message)
else:
    print("Rail blocked this run — outside the allowed window.")

That's a complete agent: **sense → decide (with a real model) → act, wrapped in a rail.** Change `sense()` and `rail()` for your own pattern, re-run, and you've built the thing the browser tool only let you diagram.

### Ship it
- **Colab:** File → Save a copy in Drive, or File → Download → `.ipynb`
- **GitHub:** upload the `.ipynb` file to any repo — GitHub renders notebooks natively, no setup needed
- Never commit a notebook with your API key typed directly into a cell. If you used the Colab-secrets method above, your key was never written into this file.

### Where this came from
- Campus Agent Lab — `mit-symposium-demo-2026.web.app/campus-agent-lab.html`
- Life Agent Builder — `mit-symposium-demo-2026.web.app/life-agent-builder.html`
- The bridge deck (case studies, the arithmetic, the eight-pattern map) — `mit-symposium-demo-2026.web.app/pradyoth-bridge-deck.html`